# 문화누리 만족활동 기반 선호분석 — 전체 재실행

이 노트북은 **Run All 한 번으로 전체 파이프라인을 순서대로 재실행**합니다.

1. 활동코드 중분류 매핑 및 만족활동 1·2·3순위 검증
2. 3:2:1 순위가중 다항 로지스틱 학습·검증
3. 100m 격자 성별×연령별 문화누리 대상자 인구 정렬
4. 격자·행정동·자치구 잠재수요, 외적 타당성, HTML 지도 생성
5. 전체 자동테스트

계산 로직을 노트북에 복사하지 않고 검증된 `src/preference_analysis` 모듈을 호출합니다. 원본 데이터는 수정하지 않으며 `data/processed` 산출물만 새로 생성합니다. 접근성·가맹점·이동시간 변수는 선호모델에 사용하지 않습니다.

> 전체 모델 학습과 60,528개 격자 지도 생성 때문에 실행에 시간이 걸릴 수 있습니다. 실행 중인 셀 왼쪽이 `[*]`이면 정상적으로 계산 중입니다.

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import FileLink, Markdown, display

def find_project_root():
    configured = os.environ.get('ORACLE_PROJECT_ROOT')
    starts = ([Path(configured).expanduser()] if configured else []) + [Path.cwd()]
    for start in starts:
        for candidate in (start, *start.parents):
            if (candidate / '.git').exists() and (candidate / 'data').is_dir():
                return candidate.resolve()
    raise FileNotFoundError('Oracle-Project 저장소를 찾지 못했습니다.')

ROOT = find_project_root()
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
LOG_DIR = ROOT / 'data/processed/preference_analysis/run_logs' / RUN_ID
LOG_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

# 전체 실행 기본값입니다. 필요할 때만 False로 바꾸세요.
BUILD_MAPS = True
RUN_EXTERNAL_VALIDATION = True
RUN_TESTS = True

def run_step(step_name, module, *module_args):
    command = [sys.executable, '-m', module, *map(str, module_args)]
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(ROOT), str(ROOT / 'src')])
    print(f'▶ {step_name}')
    print('  ', ' '.join(command))
    started = time.perf_counter()
    completed = subprocess.run(
        command, cwd=ROOT, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace'
    )
    elapsed = time.perf_counter() - started
    log_path = LOG_DIR / f'{step_name}.log'
    log_path.write_text(completed.stdout, encoding='utf-8')
    tail = '\n'.join(completed.stdout.splitlines()[-20:])
    if completed.returncode != 0:
        print(tail)
        raise RuntimeError(f'{step_name} 실패 — 전체 로그: {log_path}')
    print(f'✓ 완료 ({elapsed:.1f}초) — 로그: {log_path}')
    if tail:
        print(tail)
    return log_path

print('프로젝트:', ROOT)
print('Python:', sys.executable)
print('실행 로그:', LOG_DIR)

## 1단계 — 중분류 매핑과 만족활동 순위 검증

국민여가활동조사의 활동코드 1~88을 정책 9개 분야와 `기타·문화누리 비대응`으로 변환합니다. 향후 희망활동은 사용하지 않습니다.

In [ ]:
run_step(
    '01_중분류_매핑_검증',
    'src.preference_analysis.build_mapping',
    '--project-root', ROOT,
)

## 2단계 — 순위가중 다항 로지스틱 학습·검증

만족활동 1·2·3순위에 3:2:1 가중치를 적용합니다. 2022~2024 순차검증과 2021~2024 응답자 그룹 5-Fold Log Loss로 성별×연령 결합형과 C를 선택한 뒤, 2025년을 시간 외 평가자료로 사용합니다.

In [ ]:
run_step(
    '02_선호모델_학습_검증',
    'src.preference_analysis.train_model',
    '--project-root', ROOT,
)

## 3단계 — 100m 격자 성별×연령별 대상자 인구 정렬

기존 격자 대상자 추정치를 모델의 7개 연령구간과 성별코드에 맞춥니다. 15세 미만은 현재 선호모델 적용 대상에서 제외됩니다.

In [ ]:
run_step(
    '03_격자_성연령_인구정렬',
    'src.preference_analysis.align_population',
    '--project-root', ROOT,
)

## 4단계 — 격자·행정동·자치구 잠재수요와 지도

각 성별×연령 셀의 대상자 수에 분야별 절대 선호확률을 곱해 100m 잠재수요를 계산하고 행정동·자치구로 보존 집계합니다. 기본값은 외적 타당성 점검과 HTML 지도까지 생성합니다.

In [ ]:
spatial_args = ['--project-root', ROOT]
if not BUILD_MAPS:
    spatial_args.append('--skip-maps')
if not RUN_EXTERNAL_VALIDATION:
    spatial_args.append('--skip-external-validation')
run_step(
    '04_공간결과_외부검증_지도생성',
    'src.preference_analysis.build_spatial_outputs',
    *spatial_args,
)

## 5단계 — 전체 자동테스트

매핑, 모델 누수·확률합, 인구 총량, 격자→동→구 보존, 무자료 처리와 지도 생성을 자동검증합니다.

In [ ]:
if RUN_TESTS:
    run_step(
        '05_전체_자동테스트',
        'pytest',
        'tests/preference_analysis', '-q',
    )
else:
    print('RUN_TESTS=False — 자동테스트를 건너뛰었습니다.')

## 최종 핵심 결과

아래 셀은 전체 계산 후 핵심 성능, 공간 보존검증, 결과 규모와 지도 링크만 간결하게 보여줍니다.

In [ ]:
MODEL_DIR = ROOT / 'data/processed/preference_analysis/model'
SPATIAL_DIR = ROOT / 'data/processed/preference_analysis/spatial'

scores = pd.read_csv(MODEL_DIR / 'model_score_2024_2025.csv', encoding='utf-8-sig')
score_columns = [
    'evaluation_year', 'model', 'feature_mode', 'c_value',
    'accuracy', 'top3_accuracy', 'log_loss', 'multiclass_brier',
    'log_loss_skill_score_vs_baseline',
]
display(Markdown('### 모델 성능'))
display(scores[score_columns].round(4))

spatial_validation = pd.read_csv(
    SPATIAL_DIR / 'spatial_validation_summary_2024.csv', encoding='utf-8-sig'
)
display(Markdown('### 공간 계산 검증'))
display(spatial_validation)

metadata = json.loads(
    (SPATIAL_DIR / 'spatial_preference_run_metadata_2024.json').read_text(encoding='utf-8')
)
result_scale = pd.DataFrame([{
    '100m 격자 수': metadata['grid_count'],
    '행정동 수': metadata['dong_count'],
    '자치구 수': metadata['gu_count'],
    '15세 이상 추정 대상자': metadata['target_population_total_15plus'],
    '정책 9개 분야 잠재수요 합': metadata['policy_potential_demand_total'],
    '기타 잠재수요 합': metadata['other_potential_demand_total'],
}])
display(Markdown('### 공간 결과 규모'))
display(result_scale.round(2))

grid_map = SPATIAL_DIR / 'maps/grid_preference_demand_2024.html'
dong_map = SPATIAL_DIR / 'maps/dong_preference_demand_2024.html'
display(Markdown('### 지도 열기'))
if grid_map.exists():
    display(FileLink(str(grid_map), result_html_prefix='100m 격자 지도: '))
if dong_map.exists():
    display(FileLink(str(dong_map), result_html_prefix='행정동 지도: '))

print('✓ 전체 파이프라인 완료')
print('실행 로그:', LOG_DIR)